# Nested Booking Limits — Interactive Dashboard
Run either collapsed dashboard cell using its play button; Python code is hidden by default.


In [ ]:
#@title Nested Booking Limits — Interactive Dashboard { display-mode: "form" }
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

FARES = [('Promo',79), ('Saver',120), ('Standard',190), ('Flex',320)]
CAPACITY = 180
INITIAL_SOLD = [20,20,20,20]
INITIAL_LIMITS = [60,100,140,180]
# Identical illustrative future requests for both policies: low fares arrive first.
REQUESTS = [55,35,30,40]
sold = INITIAL_SOLD.copy()

css = widgets.HTML("""<style>
.rm{max-width:1050px;font-family:Arial,sans-serif;color:#14364a}
.rm h2{margin:5px 0}.rm h3{margin:16px 0 8px}
.rm small,.rm-note{color:#64748b;font-size:12px}
.rm-stats{display:flex;gap:10px;flex-wrap:wrap;margin:14px 0}
.rm-stat{flex:1;min-width:140px;background:#eef3f8;border-radius:10px;padding:13px}
.rm-stat b{display:block;font-size:25px;margin:3px 0}
.rm table{border-collapse:collapse;width:100%;font-size:14px;margin:12px 0}
.rm th{background:#e8f1f7;text-align:left;padding:9px}
.rm td{padding:9px;border-bottom:1px solid #dbe2e8}
.rm-good{color:#087e4e;font-weight:bold}.rm-bad{color:#b91c1c;font-weight:bold}
.rm-track{height:12px;background:#e5e7eb;border-radius:8px;overflow:hidden}
.rm-fill{height:100%;background:#08739a}
</style>""")
output = widgets.Output()
controls = widgets.Output()
buttons = []
sliders = [
    widgets.IntSlider(value=v,min=0,max=180,step=5,description=label,
        style={'description_width':'125px'},layout=widgets.Layout(width='430px'),
        continuous_update=False)
    for v,label in [(60,'Promo limit'),(100,'Saver limit'),(140,'Standard limit')]
]

def limits():
    return [w.value for w in sliders]+[CAPACITY]

def revenue(counts):
    return sum(n*fare for n,(_,fare) in zip(counts,FARES))

def available(counts, caps):
    left=CAPACITY-sum(counts)
    return [max(0,min(left,caps[i]-sum(counts[:i+1]))) for i in range(4)]

def forecast(nested):
    """Apply protection to future requests, never undo existing sales."""
    counts=sold.copy()
    accepted=[0]*4
    rejected=[0]*4
    caps=limits() if nested else [CAPACITY]*4
    for i,n in enumerate(REQUESTS):
        for _ in range(n):
            if sum(counts)<CAPACITY and (not nested or sum(counts[:i+1])<caps[i]):
                counts[i]+=1
                accepted[i]+=1
            else:
                rejected[i]+=1
    return counts,accepted,rejected

def render():
    caps=limits()
    avail=available(sold,caps)
    total=sum(sold)
    rows=''.join(
        f'<tr><td><b>{name}</b><br><small>USD {fare}</small></td>'
        f'<td>{sold[i]}</td><td>{caps[i]}</td><td>{avail[i]}</td>'
        f'<td>USD {sold[i]*fare:,}</td>'
        f'<td class="{"rm-good" if avail[i] else "rm-bad"}">'
        f'{"OPEN" if avail[i] else "CLOSED"}</td></tr>'
        for i,(name,fare) in enumerate(FARES)
    )
    protected,pa,pr=forecast(True)
    baseline,ba,br=forecast(False)
    delta=revenue(protected)-revenue(baseline)
    compare_rows=''.join(
        f'<tr><td>{name}</td><td>{pa[i]}</td><td>{ba[i]}</td>'
        f'<td>{pr[i]}</td><td>{br[i]}</td></tr>'
        for i,(name,_) in enumerate(FARES)
    )
    sign='+' if delta>0 else '-' if delta<0 else ''
    html=f"""
    <div class="rm">
      <small>ILLUSTRATIVE REVENUE MANAGEMENT DASHBOARD</small>
      <h2>Nested Booking Limits | Flight Inventory Control</h2>
      <small>180 seats · Economy cabin · D−14 (14 days before departure) · Illustrative fares and demand</small>
      <div class="rm-stats">
        <div class="rm-stat"><small>Seats sold</small><b>{total}</b></div>
        <div class="rm-stat"><small>Remaining seats</small><b>{CAPACITY-total}</b></div>
        <div class="rm-stat"><small>Load factor</small><b>{total/CAPACITY:.1%}</b></div>
        <div class="rm-stat"><small>Ticket revenue to date</small><b>USD {revenue(sold):,}</b></div>
      </div>
      <div class="rm-track"><div class="rm-fill" style="width:{100*total/CAPACITY:.2f}%"></div></div>
      <table><thead><tr><th>Fare class</th><th>Sold</th><th>Nested limit</th>
      <th>Available</th><th>Revenue to date</th><th>Status</th></tr></thead>
      <tbody>{rows}</tbody></table>
      <div class="rm-note">Available counts overlap and cannot be added. Ticket revenue excludes ancillary sales and costs.</div>
      <h3>Revenue comparison: identical future booking requests</h3>
      <small>From D−14 through departure, future requests arrive in order: Promo {REQUESTS[0]},
        Saver {REQUESTS[1]}, Standard {REQUESTS[2]}, Flex {REQUESTS[3]}.
        Only the inventory policy changes.</small>
      <div class="rm-stats">
        <div class="rm-stat"><small>WITH nested protection · final ticket revenue</small>
          <b>USD {revenue(protected):,}</b><small>{sum(protected)} total seats sold</small></div>
        <div class="rm-stat"><small>WITHOUT protection · first come, first served</small>
          <b>USD {revenue(baseline):,}</b><small>{sum(baseline)} total seats sold</small></div>
      </div>
      <div class="{"rm-good" if delta>=0 else "rm-bad"}">
      Revenue difference (protection minus no protection): {sign}USD {abs(delta):,}</div>
      <table><thead><tr><th>Future fare requests</th>
        <th>Accepted with protection</th><th>Accepted without</th>
        <th>Rejected with protection</th><th>Rejected without</th>
        </tr></thead><tbody>{compare_rows}</tbody></table>
      <div class="rm-note">Deterministic classroom scenario, not an optimized or
      statistically expected-revenue forecast. Protection can help or hurt when
      demand changes. Protection applies to future bookings; past sales remain.</div>
    </div>"""
    with output:
        clear_output(wait=True)
        display(HTML(html))
    for i,button in enumerate(buttons):
        button.disabled=avail[i]<=0

def slider_changed(change=None):
    # Cumulative limits must remain nondecreasing.
    a,b,c=sliders
    a.max=b.value
    b.min=a.value
    b.max=c.value
    c.min=b.value
    render()

for slider in sliders:
    slider.observe(slider_changed,names='value')

def book(i):
    if available(sold,limits())[i]>0:
        sold[i]+=1
        render()

for i,(name,fare) in enumerate(FARES):
    button=widgets.Button(description=f'+1 {name} (USD {fare})',
                          layout=widgets.Layout(width='205px'))
    button.on_click(lambda _,i=i:book(i))
    buttons.append(button)

reset=widgets.Button(description='↻ Reset example',button_style='info',
                     layout=widgets.Layout(width='185px'))
def reset_all(_):
    sold[:]=INITIAL_SOLD
    sliders[0].value=0
    sliders[1].value=0
    sliders[2].value=INITIAL_LIMITS[2]
    sliders[1].value=INITIAL_LIMITS[1]
    sliders[0].value=INITIAL_LIMITS[0]
    render()
reset.on_click(reset_all)

display(css,output,
    widgets.HTML('<b>Accept one booking:</b>'),
    widgets.HBox(buttons,layout=widgets.Layout(flex_flow='row wrap',gap='8px')),
    widgets.HTML('<b>Adjust nested booking limits (cumulative):</b>'),
    widgets.VBox(sliders),reset)
slider_changed()


## Activity 2 — Seat-Inventory Decision

**Separate scenario:** A **180-seat** flight already has **160 bookings** (20 seats remain). Today, **15 requests at $120** arrive. Later, there is a **70% chance of 15 requests at $300**, otherwise no later demand. Assume all bookings are honoured. Ignore the revenue from the 160 existing bookings because it is the same in both options.

- **A — Accept all 15** low-fare requests today (only 5 seats remain for later).
- **B — Accept 5** low-fare requests today and protect 15 seats for later.

**Questions**

1. For A and B, calculate revenue today, expected later revenue, and expected total *incremental* revenue.
2. At 70%, which option earns more expected revenue?
3. Adjust the probability in the simulator. At what probability are you indifferent between A and B?

*The Activity 2 simulator is independent of the main 80-bookings-sold dashboard above.*


In [ ]:
#@title Activity 2 — Expected Revenue Comparison { display-mode: "form" }
import ipywidgets as widgets
from IPython.display import HTML, display, clear_output

chance = widgets.IntSlider(value=70, min=0, max=100, step=5,
    description='Later demand probability', style={'description_width':'190px'},
    layout=widgets.Layout(width='550px'), continuous_update=False)
result = widgets.Output()

def show_activity(change=None):
    p = chance.value / 100
    today_A = 15 * 120
    today_B = 5 * 120
    later_A = p * 5 * 300
    later_B = p * 15 * 300
    total_A = today_A + later_A
    total_B = today_B + later_B
    if abs(total_A - total_B) < 0.001:
        preferred = 'Both decisions have equal expected revenue.'
    else:
        preferred = 'B: Protect 15 seats' if total_B > total_A else 'A: Accept all 15'
    html = f"""
    <div style="font-family:Arial,sans-serif;max-width:840px;color:#0b3650">
      <h3 style="margin:8px 0">Activity 2 · Expected Revenue</h3>
      <div style="color:#64748b;font-size:13px">180 seats · 160 already booked · 20 remaining ·
        15 low-fare requests now (USD 120) · 15 potential high-fare requests later (USD 300)</div>
      <p>Probability of later high-fare demand: <b>{chance.value}%</b></p>
      <table style="width:100%;border-collapse:collapse;text-align:right">
      <thead><tr style="background:#e8f1f7">
        <th style="text-align:left;padding:10px">Decision</th>
        <th style="padding:10px">Revenue today</th>
        <th style="padding:10px">Expected later</th>
        <th style="padding:10px">Expected incremental total</th>
      </tr></thead>
      <tbody>
      <tr><td style="text-align:left;padding:10px;border-bottom:1px solid #ddd">A · Accept all 15</td>
        <td style="padding:10px;border-bottom:1px solid #ddd">USD {today_A:,.0f}</td>
        <td style="padding:10px;border-bottom:1px solid #ddd">USD {later_A:,.0f}</td>
        <td style="padding:10px;border-bottom:1px solid #ddd"><b>USD {total_A:,.0f}</b></td></tr>
      <tr><td style="text-align:left;padding:10px;border-bottom:1px solid #ddd">B · Protect 15</td>
        <td style="padding:10px;border-bottom:1px solid #ddd">USD {today_B:,.0f}</td>
        <td style="padding:10px;border-bottom:1px solid #ddd">USD {later_B:,.0f}</td>
        <td style="padding:10px;border-bottom:1px solid #ddd"><b>USD {total_B:,.0f}</b></td></tr>
      </tbody></table>
      <p><b>Higher expected revenue:</b> {preferred}</p>
      <p style="color:#64748b;font-size:12px">Incremental ticket revenue only;
      excludes revenue from the 160 bookings already made.</p>
    </div>"""
    with result:
        clear_output(wait=True)
        display(HTML(html))

chance.observe(show_activity, names='value')
reset_activity = widgets.Button(description='Reset probability to 70%',
    layout=widgets.Layout(width='225px'))
reset_activity.on_click(lambda _: setattr(chance, 'value', 70))
display(chance, reset_activity, result)
show_activity()
